In [1]:
from functools import reduce

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
dfs = []

for subject in ["mathematics", "reading", "science"]:
    df = pd.read_csv(f"../../data/raw/pisa_{subject}_2000_2022.csv")
    df["subject"] = subject
    dfs.append(df)

df_pisa = pd.concat(dfs, ignore_index=True)

In [3]:
# TODO Considerar formato ancho para modelado (formato largo es mejor para tratar asignaturas por separado)

In [26]:
dfs = []

for subject in ["mathematics", "reading", "science"]:
    df = pd.read_csv(f"../../data/raw/pisa_{subject}_2000_2022.csv")

    # Renombrar todas las columnas excepto las claves
    df = df.rename(
        columns={
            col: f"{subject}_{col}"
            for col in df.columns
            if col not in ["country", "year"]
        }
    )

    dfs.append(df)

df_pisa_ancho = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=["country", "year"],
        how="outer"
    ),
    dfs
)

In [ ]:
# Algunos países tienen 2015 separado del resto en un país con nomenclatura: [nombre del país] (2015). Por ejemplo, Argentina no tiene
# valores en 2015, y Argentina (2015) tiene todo nulo excepto 2015. Se combinan

# Se elimina el (2015) del nombre de la columna
df_pisa["country"] = df_pisa["country"].str.replace(
    r" \(2015\)$", "", regex=True
)

# Ahora que tienen el mismo nombre, se agrupan y se usa .first(), que coge el primer valor no nulo (que será el de la columna
# original excepto en 2015).
df_pisa = (
    df_pisa
    .groupby(["country", "year", "subject"], as_index=False)
    .first()
)

In [21]:
nulls_country_year = (
    df_pisa
    .groupby(["country", "year", "subject"])
    .apply(lambda x: x.isna().sum().sum())
    .reset_index(name="n_nulls")
)

nulls_country_year

,country,year,subject,n_nulls
0,Albania,2000,mathematics,18
1,Albania,2000,reading,0
2,Albania,2000,science,18
3,Albania,2003,mathematics,18
4,Albania,2003,reading,18
...,...,...,...,...
2227,Viet Nam,2018,reading,18
2228,Viet Nam,2018,science,18
2229,Viet Nam,2022,mathematics,0
2230,Viet Nam,2022,reading,0
